# Chapter 13 Walkthrough: Mastering Jupyter Notebooks & Google Colab Engineering

**Book Reference:** Chapter 13, Sections 13.1 – 13.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every file demo runs inside `tempfile.TemporaryDirectory()` so zero temporary files are left behind, and Colab-specific commands are modeled safely without mounting a live Drive.

## Section 13.3.1 — `.ipynb` JSON Anatomy & Hidden-State Out-of-Order Audit

In [ ]:
import json
from pathlib import Path
import tempfile

sample_nb = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {},
    "cells": [
        {"cell_type": "markdown", "source": ["# Dengue Audit\n"]},
        {
            "cell_type": "code",
            "execution_count": 5,
            "source": ["threshold = 38.0\n"],
            "outputs": [],
        },
        {
            "cell_type": "code",
            "execution_count": 2,
            "source": ["print(threshold)\n"],
            "outputs": [{"output_type": "stream", "text": ["38.0\n"]}],
        },
    ],
}

def is_out_of_order(nb_dict):
    prev = 0
    for cell in nb_dict["cells"]:
        if cell["cell_type"] != "code":
            continue
        ec = cell["execution_count"]
        if ec is None or ec <= prev:
            return True
        prev = ec
    return False

with tempfile.TemporaryDirectory() as tmp:
    nb_file = Path(tmp) / "analysis.ipynb"
    nb_file.write_text(json.dumps(sample_nb, indent=2), encoding="utf-8")
    loaded = json.loads(nb_file.read_text(encoding="utf-8"))

counts = [
    c["execution_count"] for c in loaded["cells"]
    if c["cell_type"] == "code"
]
print("Code cell counts:", counts)
print("Hidden-state risk (out of order):", is_out_of_order(loaded))


## Section 13.3.2 — Portable Colab/Local Path Resolver & Magic Awareness

In [ ]:
from pathlib import Path
import tempfile

# Colab-only reference (never invoke live inside portable modules):
# from google.colab import drive; drive.mount('/content/drive')
# Interactive magics like '%timeit' or '%%writefile' are IPython-only.

def find_dataset(filename, drive_dir=None, local_dir=None):
    for folder in (drive_dir, local_dir):
        if folder is not None:
            candidate = Path(folder) / filename
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Missing dataset: {filename}")

with tempfile.TemporaryDirectory() as tmp:
    sim_drive = Path(tmp) / "colab_drive"
    sim_local = Path(tmp) / "local_data"
    sim_drive.mkdir()
    sim_local.mkdir()
    (sim_local / "mch_cohort.csv").write_text("id,hb\n1,11.2\n")

    resolved = find_dataset("mch_cohort.csv", sim_drive, sim_local)
    print("Resolved source folder:", resolved.parent.name)
    print("First line:", resolved.read_text().splitlines()[0])


## Section 13.3.3 — Crash-Proof Atomic JSON Checkpointing

In [ ]:
import json
from pathlib import Path
import tempfile

def write_checkpoint(ckpt_path, state_dict):
    target = Path(ckpt_path)
    tmp_path = target.with_suffix(".tmp")
    with open(tmp_path, "w", encoding="utf-8") as out_f:
        json.dump(state_dict, out_f, indent=2)
    tmp_path.replace(target)

with tempfile.TemporaryDirectory() as tmp:
    ckpt = Path(tmp) / "train_state.json"
    write_checkpoint(ckpt, {"epoch": 1, "val_auc": 0.81})
    write_checkpoint(ckpt, {"epoch": 2, "val_auc": 0.86})

    restored = json.loads(ckpt.read_text(encoding="utf-8"))
    leftover_tmp = ckpt.with_suffix(".tmp").exists()
    print("Restored checkpoint:", restored)
    print("Leftover .tmp exists:", leftover_tmp)


## Section 13.3.4 — Isolated RNG Reproducibility (`random.Random(seed)`)

In [ ]:
import random

def split_indices(n_samples, test_ratio=0.2, seed=42):
    rng = random.Random(seed)
    indices = list(range(n_samples))
    rng.shuffle(indices)
    n_test = int(n_samples * test_ratio)
    return indices[n_test:], indices[:n_test]

random.seed(999)
before_global = random.getstate()
train_a, test_a = split_indices(10, test_ratio=0.2, seed=42)
after_global = random.getstate()

random.seed(12345)
train_b, test_b = split_indices(10, test_ratio=0.2, seed=42)

print("Test fold indices:", test_a)
print("Identical across global reseeds:", test_a == test_b)
print("Global RNG untouched:", before_global == after_global)


## Section 13.3.5 — Break It On Purpose! (Notebook & Colab Error Anatomy)

In [ ]:
# Intentional Bug #1: FileNotFoundError — unmounted Drive / missing dataset
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    unmounted_path = Path(tmp) / "MyDrive" / "raw_cohort.csv"
    with open(unmounted_path, "r", encoding="utf-8") as f:
        header = f.readline()


In [ ]:
# Intentional Bug #2: json.JSONDecodeError — truncated checkpoint file
import json
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    corrupt_ckpt = Path(tmp) / "train_state.json"
    corrupt_ckpt.write_text('{"epoch": 8, "val_auc": ', encoding="utf-8")
    with open(corrupt_ckpt, "r", encoding="utf-8") as f:
        state = json.load(f)


In [ ]:
# Intentional Bug #3: KeyError — malformed notebook missing 'cells'
import json
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    bad_nb_path = Path(tmp) / "malformed.ipynb"
    bad_nb_path.write_text('{"nbformat": 4}', encoding="utf-8")
    with open(bad_nb_path, "r", encoding="utf-8") as f:
        nb_data = json.load(f)
    cell_count = len(nb_data["cells"])


## Section 13.4 — Worked Problem-Structuring Blueprint: Notebook Preflight & Clean Export

In [ ]:
import copy
import json
from pathlib import Path
import tempfile

def preflight_and_clean_notebook(in_nb, out_nb):
    """Audit execution order, strip outputs, and save atomically."""
    with open(in_nb, "r", encoding="utf-8") as in_f:
        raw = json.load(in_f)

    prev = 0
    out_of_order = False
    had_outputs = False
    cleaned = copy.deepcopy(raw)

    for cell in cleaned.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        ec = cell.get("execution_count")
        if ec is None or ec <= prev:
            out_of_order = True
        else:
            prev = ec
        if cell.get("outputs"):
            had_outputs = True
        cell["outputs"] = []
        cell["execution_count"] = None

    target = Path(out_nb)
    tmp_path = target.with_suffix(".tmp")
    with open(tmp_path, "w", encoding="utf-8") as out_f:
        json.dump(cleaned, out_f, indent=2)
    tmp_path.replace(target)

    return {"out_of_order": out_of_order, "had_outputs": had_outputs}

with tempfile.TemporaryDirectory() as tmp:
    raw_nb = Path(tmp) / "draft.ipynb"
    git_nb = Path(tmp) / "clean.ipynb"
    draft = {
        "nbformat": 4,
        "nbformat_minor": 5,
        "metadata": {},
        "cells": [
            {
                "cell_type": "code",
                "execution_count": 4,
                "source": ["x = 7\n"],
                "outputs": [{"output_type": "stream", "text": ["7\n"]}],
            },
            {
                "cell_type": "code",
                "execution_count": 2,
                "source": ["y = x + 1\n"],
                "outputs": [],
            },
        ],
    }
    raw_nb.write_text(json.dumps(draft, indent=2), encoding="utf-8")
    report = preflight_and_clean_notebook(raw_nb, git_nb)
    saved = json.loads(git_nb.read_text(encoding="utf-8"))
    print("Preflight report:", report)
    print("Clean cell 0 outputs:", saved["cells"][0]["outputs"])
